<a href="https://colab.research.google.com/github/KrithikChidM/BigDataAndDataEngineering/blob/main/hadoopdataanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk postgresql postgresql-contrib

!pip install -q pyspark==4.0.0 kafka-python psycopg2-binary

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-common at-spi2-core fonts-dejavu-core fonts-dejavu-extra
  fonts-dejavu-mono gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0t64 libatspi2.0-0t64
  libcommon-sense-perl libgail-common libgail18t64 libgtk2.0-0t64
  libgtk2.0-bin libgtk2.0-common libjson-perl libjson-xs-perl librsvg2-common
  libtypes-serialiser-perl libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  logrotate openjdk-17-jdk-headless openjdk-17-jre openjdk-17-jre-headless
  postgresql-16 postgresql-client-16 postgresql-client-common
  postgresql-common session-migration ssl-cert sysstat x11-utils
Suggested packages:
  gvfs l

In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [ ]:
!java -version
import pyspark
print('PySpark:', pyspark.__version__)

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-24.04-Ubuntu, mixed mode, sharing)
PySpark: 4.0.0


In [ ]:
!wget -q https://archive.apache.org/dist/kafka/4.0.0/kafka_2.13-4.0.0.tgz
!tar -xzf kafka_2.13-4.0.0.tgz

In [ ]:
KAFKA_HOME = "/content/kafka_2.13-4.0.0"

In [ ]:
!rm /content/kafka_2.13-4.0.0.tgz
!ls /content/kafka_2.13-4.0.0/bin/

connect-distributed.sh		 kafka-jmx.sh
connect-mirror-maker.sh		 kafka-leader-election.sh
connect-plugin-path.sh		 kafka-log-dirs.sh
connect-standalone.sh		 kafka-metadata-quorum.sh
kafka-acls.sh			 kafka-metadata-shell.sh
kafka-broker-api-versions.sh	 kafka-producer-perf-test.sh
kafka-client-metrics.sh		 kafka-reassign-partitions.sh
kafka-cluster.sh		 kafka-replica-verification.sh
kafka-configs.sh		 kafka-run-class.sh
kafka-console-consumer.sh	 kafka-server-start.sh
kafka-console-producer.sh	 kafka-server-stop.sh
kafka-console-share-consumer.sh  kafka-share-groups.sh
kafka-consumer-groups.sh	 kafka-storage.sh
kafka-consumer-perf-test.sh	 kafka-streams-application-reset.sh
kafka-delegation-tokens.sh	 kafka-topics.sh
kafka-delete-records.sh		 kafka-transactions.sh
kafka-dump-log.sh		 kafka-verifiable-consumer.sh
kafka-e2e-latency.sh		 kafka-verifiable-producer.sh
kafka-features.sh		 trogdor.sh
kafka-get-offsets.sh		 windows
kafka-groups.sh


In [ ]:
!cd /content/kafka_2.13-4.0.0 && bin/kafka-storage.sh random-uuid

lDkewewoSk22vsu0WJUbRg


In [ ]:
import subprocess

cluster_id = subprocess.check_output(
    [f"{KAFKA_HOME}/bin/kafka-storage.sh", "random-uuid"],
    text=True
).strip()

subprocess.run([
    f"{KAFKA_HOME}/bin/kafka-storage.sh",
    "format",
    "--standalone",
    "-t", cluster_id,
    "-c", f"{KAFKA_HOME}/config/server.properties"
], check=True)

CompletedProcess(args=['/content/kafka_2.13-4.0.0/bin/kafka-storage.sh', 'format', '--standalone', '-t', 'Mmo7Smt7SCS-o2KW7K39Uw', '-c', '/content/kafka_2.13-4.0.0/config/server.properties'], returncode=0)

In [ ]:
!nohup /content/kafka_2.13-4.0.0/bin/kafka-server-start.sh \
/content/kafka_2.13-4.0.0/config/server.properties \
> /content/kafka.log 2>&1 &

In [ ]:
import time
import socket

for i in range(30):
    try:
        with socket.create_connection(("localhost", 9092), timeout=1):
            print("Kafka broker is running on port 9092")
            break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError("Kafka did not start. Check /content/kafka.log")

Kafka broker is running on port 9092


In [ ]:
!tail -5 /content/kafka.log

[2026-11-01 17:29:29,865] INFO [BrokerServer id=1] Transition from STARTING to STARTED (kafka.server.BrokerServer)
[2026-11-01 17:29:29,867] INFO Kafka version: 4.0.0 (org.apache.kafka.common.utils.AppInfoParser)
[2026-11-01 17:29:29,867] INFO Kafka commitId: 985bc99521dd22bb (org.apache.kafka.common.utils.AppInfoParser)
[2026-11-01 17:29:29,867] INFO Kafka startTimeMs: 1793554169866 (org.apache.kafka.common.utils.AppInfoParser)
[2026-11-01 17:29:29,870] INFO [KafkaRaftServer nodeId=1] Kafka Server started (kafka.server.KafkaRaftServer)


In [ ]:
! /content/kafka_2.13-4.0.0/bin/kafka-topics.sh \
--create \
--topic transactions \
--bootstrap-server localhost:9092 \
--partitions 1 \
--replication-factor 1

Created topic transactions.


In [ ]:
! /content/kafka_2.13-4.0.0/bin/kafka-topics.sh \
--list \
--bootstrap-server localhost:9092

transactions


In [ ]:
! /content/kafka_2.13-4.0.0/bin/kafka-topics.sh \
--describe \
--topic transactions \
--bootstrap-server localhost:9092

Topic: transactions	TopicId: jVpuH-PxRWmyMEkoAC88ng	PartitionCount: 1	ReplicationFactor: 1	Configs: segment.bytes=1073741824
	Topic: transactions	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 


```
{
  "transaction_id": "T101",
  "product_id": "P101",
  "quantity": 2,
  "price": 100.0,
  "timestamp": "2026-11-01T10:30:00"
}
```

In [ ]:
from kafka import KafkaProducer
import json
import time
import random
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)

def send_transaction(transaction_id, product_id, quantity, price):
    transaction = {
        "transaction_id": transaction_id,
        "product_id": product_id,
        "quantity": quantity,
        "price": float(price),
        "timestamp": datetime.now().isoformat()
    }

    producer.send("transactions", value=transaction)
    producer.flush()

    print("Published:", transaction)

def publish_continuously():
    count = 1

    while True:
        product_id = random.choice(["P101", "P102", "P103"])
        quantity = random.randint(1, 5)
        price = random.choice([100, 150, 200])

        send_transaction(
            f"T{count:03d}",
            product_id,
            quantity,
            price
        )

        count += 1
        time.sleep(3)

In [ ]:
send_transaction("T101", "P101", 2, 100)
send_transaction("T102", "P102", 1, 200)
send_transaction("T103", "P101", 3, 100)
send_transaction("T104", "P103", 2, 150)

Published: {'transaction_id': 'T101', 'product_id': 'P101', 'quantity': 2, 'price': 100.0, 'timestamp': '2026-11-01T17:32:10.942424'}
Published: {'transaction_id': 'T102', 'product_id': 'P102', 'quantity': 1, 'price': 200.0, 'timestamp': '2026-11-01T17:32:10.995388'}
Published: {'transaction_id': 'T103', 'product_id': 'P101', 'quantity': 3, 'price': 100.0, 'timestamp': '2026-11-01T17:32:11.003481'}
Published: {'transaction_id': 'T104', 'product_id': 'P103', 'quantity': 2, 'price': 150.0, 'timestamp': '2026-11-01T17:32:11.007976'}


In [ ]:
!timeout 10 /content/kafka_2.13-4.0.0/bin/kafka-console-consumer.sh \
--bootstrap-server localhost:9092 \
--topic transactions \
--from-beginning

{"transaction_id": "T101", "product_id": "P101", "quantity": 2, "price": 100.0, "timestamp": "2026-11-01T17:32:10.942424"}
{"transaction_id": "T102", "product_id": "P102", "quantity": 1, "price": 200.0, "timestamp": "2026-11-01T17:32:10.995388"}
{"transaction_id": "T103", "product_id": "P101", "quantity": 3, "price": 100.0, "timestamp": "2026-11-01T17:32:11.003481"}
{"transaction_id": "T104", "product_id": "P103", "quantity": 2, "price": 150.0, "timestamp": "2026-11-01T17:32:11.007976"}
Processed a total of 4 messages


In [ ]:
!service postgresql start

[ OK ]


In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';"

ALTER ROLE


In [ ]:
!sudo -u postgres psql -c "CREATE DATABASE transaction_db;"

CREATE DATABASE


In [ ]:
%%bash
sudo -u postgres psql -d transaction_db -c "
CREATE TABLE IF NOT EXISTS product_sales (
    product_id VARCHAR(50) PRIMARY KEY,
    total_quantity BIGINT,
    total_sales DOUBLE PRECISION,
    latest_transaction_time TIMESTAMP,
    last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"

CREATE TABLE


In [ ]:
!timeout 20 sudo -u postgres psql -d transaction_db -c "\dt"

             List of relations
 Schema |     Name      | Type  |  Owner   
--------+---------------+-------+----------
 public | product_sales | table | postgres
(1 row)

>8

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RealTimeTransactionAnalytics")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
    )
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)

Spark Version: 4.0.0


In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])

In [ ]:
from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    sum,
    max,
    round
)

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 100)
    .load()
)

In [ ]:
transactions_df = (
    kafka_df
    .select(
        from_json(
            col("value").cast("string"),
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
    .withColumn(
        "transaction_time",
        to_timestamp(col("timestamp"))
    )
    .filter(
        col("transaction_id").isNotNull() &
        col("product_id").isNotNull() &
        col("quantity").isNotNull() &
        col("price").isNotNull() &
        col("transaction_time").isNotNull()
    )
)

In [ ]:
sales_df = transactions_df.withColumn(
    "sales",
    col("quantity") * col("price")
)

In [ ]:
sales_df

DataFrame[transaction_id: string, product_id: string, quantity: int, price: double, timestamp: string, transaction_time: timestamp, sales: double]

In [ ]:
aggregated_df = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity"),
        round(sum("sales"), 2).alias("total_sales"),
        max("transaction_time").alias("latest_transaction_time")
    )
)

Spark Structured Streaming provides three common output modes:
Output mode	Meaning
\begin{center}
  \begin{tabular}{ |c|c|\\ }
    \hline
    Append & 	Writes only newly appended rows \\
    \hline
    Update & 	Writes rows whose aggregate values have changed \\
    \hline
    Complete & 	Writes the entire aggregation result every trigger \\
    \hline
  \end{tabular}
\end{center}\


In [ ]:
import psycopg2

def write_to_postgres(batch_df, batch_id):

    rows = batch_df.collect()

    if not rows:
        return

    connection = psycopg2.connect(
        host="localhost",
        port=5432,
        database="transaction_db",
        user="postgres",
        password="postgres"
    )

    cursor = connection.cursor()

    query = """
        INSERT INTO product_sales (
            product_id,
            total_quantity,
            total_sales,
            latest_transaction_time,
            last_updated
        )
        VALUES (%s, %s, %s, %s, CURRENT_TIMESTAMP)

        ON CONFLICT (product_id)
        DO UPDATE SET
            total_quantity = EXCLUDED.total_quantity,
            total_sales = EXCLUDED.total_sales,
            latest_transaction_time = EXCLUDED.latest_transaction_time,
            last_updated = CURRENT_TIMESTAMP;
    """

    values = [
        (
            row["product_id"],
            row["total_quantity"],
            float(row["total_sales"]),
            row["latest_transaction_time"]
        )
        for row in rows
    ]

    cursor.executemany(query, values)

    connection.commit()

    cursor.close()
    connection.close()

    print(f"Batch {batch_id}: {len(rows)} product records updated")

In [ ]:
query = (
    aggregated_df
    .writeStream
    .outputMode("update")
    .foreachBatch(write_to_postgres)
    .option(
        "checkpointLocation",
        "/content/spark_checkpoint"
    )
    .trigger(processingTime="5 seconds")
    .start()
)

print("Spark Structured Streaming started!")

Spark Structured Streaming started!


In [ ]:
print("Active:", query.isActive)
print("Status:", query.status)

Active: True
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}


In [ ]:
import time

time.sleep(10)

print(query.lastProgress)

{
  "id" : "58f57bf9-a16b-47ff-91b7-0b88a62e2a0d",
  "runId" : "db03d47e-dba9-4172-980f-4c354fed8490",
  "name" : null,
  "timestamp" : "2026-11-01T17:46:50.000Z",
  "batchId" : 1,
  "batchDuration" : 10,
  "numInputRows" : 0,
  "inputRowsPerSecond" : 0.0,
  "processedRowsPerSecond" : 0.0,
  "durationMs" : {
    "latestOffset" : 9,
    "triggerExecution" : 10
  },
  "stateOperators" : [ {
    "operatorName" : "stateStoreSave",
    "numRowsTotal" : 3,
    "numRowsUpdated" : 0,
    "allUpdatesTimeMs" : 440,
    "numRowsRemoved" : 0,
    "allRemovalsTimeMs" : 2,
    "commitTimeMs" : 202,
    "memoryUsedBytes" : 1208,
    "numRowsDroppedByWatermark" : 0,
    "numShufflePartitions" : 2,
    "numStateStoreInstances" : 2,
    "customMetrics" : {
      "loadedMapCacheHitCount" : 0,
      "loadedMapCacheMissCount" : 0,
      "stateOnCurrentVersionSizeBytes" : 920
    }
  } ],
  "sources" : [ {
    "description" : "KafkaV2[Subscribe[transactions]]",
    "startOffset" : {
      "transactions" : {

In [ ]:
def display_database_results():

    connection = psycopg2.connect(
        host="localhost",
        port=5432,
        database="transaction_db",
        user="postgres",
        password="postgres"
    )

    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            product_id,
            total_quantity,
            total_sales,
            latest_transaction_time,
            last_updated
        FROM product_sales
        ORDER BY product_id;
    """)

    results = cursor.fetchall()

    for row in results:
        print(row)

    cursor.close()
    connection.close()
display_database_results()

('P101', 5, 500.0, datetime.datetime(2026, 11, 1, 17, 32, 11, 3481), datetime.datetime(2026, 11, 1, 17, 46, 20, 90790))
('P102', 1, 200.0, datetime.datetime(2026, 11, 1, 17, 32, 10, 995388), datetime.datetime(2026, 11, 1, 17, 46, 20, 90790))
('P103', 2, 300.0, datetime.datetime(2026, 11, 1, 17, 32, 11, 7976), datetime.datetime(2026, 11, 1, 17, 46, 20, 90790))


In [ ]:
send_transaction("T105", "P101", 2, 100)

Published: {'transaction_id': 'T105', 'product_id': 'P101', 'quantity': 2, 'price': 100.0, 'timestamp': '2026-11-01T17:48:15.861614'}


In [ ]:
time.sleep(10)

display_database_results()

('P101', 7, 700.0, datetime.datetime(2026, 11, 1, 17, 48, 15, 861614), datetime.datetime(2026, 11, 1, 17, 48, 21, 664303))
('P102', 1, 200.0, datetime.datetime(2026, 11, 1, 17, 32, 10, 995388), datetime.datetime(2026, 11, 1, 17, 46, 20, 90790))
('P103', 2, 300.0, datetime.datetime(2026, 11, 1, 17, 32, 11, 7976), datetime.datetime(2026, 11, 1, 17, 46, 20, 90790))


In [ ]:
publish_continuously()

Published: {'transaction_id': 'T001', 'product_id': 'P101', 'quantity': 3, 'price': 100.0, 'timestamp': '2026-11-01T17:49:42.427474'}
Published: {'transaction_id': 'T002', 'product_id': 'P101', 'quantity': 1, 'price': 100.0, 'timestamp': '2026-11-01T17:49:45.443651'}
Batch 2: 1 product records updated
Published: {'transaction_id': 'T003', 'product_id': 'P103', 'quantity': 4, 'price': 100.0, 'timestamp': '2026-11-01T17:49:48.454663'}
Batch 3: 2 product records updated
Published: {'transaction_id': 'T004', 'product_id': 'P103', 'quantity': 2, 'price': 200.0, 'timestamp': '2026-11-01T17:49:51.458595'}
Published: {'transaction_id': 'T005', 'product_id': 'P102', 'quantity': 5, 'price': 150.0, 'timestamp': '2026-11-01T17:49:54.464663'}
Batch 4: 2 product records updated
Published: {'transaction_id': 'T006', 'product_id': 'P101', 'quantity': 2, 'price': 150.0, 'timestamp': '2026-11-01T17:49:57.470688'}
Published: {'transaction_id': 'T007', 'product_id': 'P103', 'quantity': 3, 'price': 150.0, 

KeyboardInterrupt: 

In [ ]:
display_database_results()

('P101', 13, 1400.0, datetime.datetime(2026, 11, 1, 17, 49, 57, 470688), datetime.datetime(2026, 11, 1, 17, 50, 0, 737899))
('P102', 6, 950.0, datetime.datetime(2026, 11, 1, 17, 49, 54, 464663), datetime.datetime(2026, 11, 1, 17, 49, 56, 642984))
('P103', 11, 1550.0, datetime.datetime(2026, 11, 1, 17, 50, 0, 475401), datetime.datetime(2026, 11, 1, 17, 50, 6, 362020))


In [ ]:
! /content/kafka_2.13-4.0.0/bin/kafka-topics.sh \
--list \
--bootstrap-server localhost:9092

__consumer_offsets
transactions


In [ ]:
!timeout 10 sudo -u postgres psql -d transaction_db -c \
"SELECT * FROM product_sales ORDER BY product_id;"

 product_id | total_quantity | total_sales |  latest_transaction_time   |         last_updated        
------------+----------------+-------------+----------------------------+------- ---------------------
 P101       |             13 |        1400 | 2026-11-01 17:49:57.470688 | 2026-1 1-01 17:50:00.737899
 P102       |              6 |         950 | 2026-11-01 17:49:54.464663 | 2026-1 1-01 17:49:56.642984
 P103       |             11 |        1550 | 2026-11-01 17:50:00.475401 | 2026-1 1-01 17:50:06.36202
(3 rows)

>8

In [ ]:
!timeout 10 service postgresql status

16/main (port 5432): online


In [ ]:
!tail -5 /content/kafka.log

[2026-11-01 17:32:34,917] INFO [GroupCoordinator id=1 topic=__consumer_offsets partition=6] Preparing to rebalance group console-consumer-83239 in state PreparingRebalance with old generation 1 (reason: explicit `LeaveGroup` request for (console-consumer-fdbc3742-5079-4b1d-8e1a-5463c7e9c10e) members.). (org.apache.kafka.coordinator.group.GroupMetadataManager)
[2026-11-01 17:32:34,917] INFO [GroupCoordinator id=1 topic=__consumer_offsets partition=6] Group console-consumer-83239 with generation 2 is now empty. (org.apache.kafka.coordinator.group.GroupMetadataManager)
[2026-11-01 17:39:28,551] INFO [NodeToControllerChannelManager id=1 name=registration] Node 1 disconnected. (org.apache.kafka.clients.NetworkClient)
[2026-11-01 17:42:29,479] INFO [NodeToControllerChannelManager id=1 name=forwarding] Node 1 disconnected. (org.apache.kafka.clients.NetworkClient)
[2026-11-01 17:42:30,571] INFO [GroupCoordinator id=1 topic=__consumer_offsets partition=6] Generated 1 tombstone records while cle

In [ ]:
print("Spark Active:", query.isActive)

Spark Active: True


In [ ]:
print("Streaming Status:", query.status)

Streaming Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}


In [ ]:
print("Last Progress:", query.lastProgress)

Last Progress: {
  "id" : "58f57bf9-a16b-47ff-91b7-0b88a62e2a0d",
  "runId" : "db03d47e-dba9-4172-980f-4c354fed8490",
  "name" : null,
  "timestamp" : "2026-11-01T17:53:10.000Z",
  "batchId" : 7,
  "batchDuration" : 39,
  "numInputRows" : 0,
  "inputRowsPerSecond" : 0.0,
  "processedRowsPerSecond" : 0.0,
  "durationMs" : {
    "latestOffset" : 37,
    "triggerExecution" : 39
  },
  "stateOperators" : [ {
    "operatorName" : "stateStoreSave",
    "numRowsTotal" : 3,
    "numRowsUpdated" : 0,
    "allUpdatesTimeMs" : 80,
    "numRowsRemoved" : 0,
    "allRemovalsTimeMs" : 0,
    "commitTimeMs" : 212,
    "memoryUsedBytes" : 1696,
    "numRowsDroppedByWatermark" : 0,
    "numShufflePartitions" : 2,
    "numStateStoreInstances" : 2,
    "customMetrics" : {
      "loadedMapCacheHitCount" : 24,
      "loadedMapCacheMissCount" : 0,
      "stateOnCurrentVersionSizeBytes" : 832
    }
  } ],
  "sources" : [ {
    "description" : "KafkaV2[Subscribe[transactions]]",
    "startOffset" : {
      "t

In [ ]:
query.stop()

In [ ]:
producer.close()